In [2]:
# MULTIMODAL PREPROCESSING FOR TSFNet
# Preprocessing with 11seg for fNIRS just like the apper and removing the normalization

import os
import gc
import numpy as np
import scipy.io as sio
from scipy.signal import butter, filtfilt
from scipy.interpolate import CloughTocher2DInterpolator
from scipy.spatial import Delaunay
import mne
import mne_icalabel
import pywt
import scipy.stats as stats
import pandas as pd
from scipy.interpolate import UnivariateSpline
import time
import datetime
import json
import warnings
warnings.filterwarnings('ignore')

# CONFIGURATION

BASE_DATASET_PATH = r"E:\kntu\Thesis\codes\DATA"  # NEW: matches user's actual dataset folder
OUTPUT_PATH = r"E:\kntu\Thesis\codes\Results\MULTIMODAL_PREPROCESSED"  # NEW: matches user's actual output folder

# WINDOW PARAMETERS 
EEG_WINDOW_LEN = 3.0
EEG_STEP = 1.0
EEG_EPOCH_TMIN = -2.0
EEG_EPOCH_TMAX = 10.0

FNIRS_WINDOW_LEN = 3.0
FNIRS_STEP = 1.0
FNIRS_NUM_SEGMENTS = 11

SFREQ_EEG = 200.0
SFREQ_FNIRS = 10.0

# MARKERS
EEG_TRIAL_LABEL_MAP = {112: 0, 128: 1, 144: 2}  # 0-back, 2-back, 3-back

# LOAD EEG DATA
def load_eeg_data(eeg_folder):
    cnt_path = os.path.join(eeg_folder, 'cnt_nback.mat')
    mrk_path = os.path.join(eeg_folder, 'mrk_nback.mat')
    mnt_path = os.path.join(eeg_folder, 'mnt_nback.mat')
    
    print(f"loading EEG from: {cnt_path}")
    
    mat_cnt = sio.loadmat(cnt_path, struct_as_record=False, squeeze_me=True)
    real_cnt_keys = [k for k in mat_cnt.keys() if not k.startswith('__')]
    cnt_key = real_cnt_keys[0]
    cnt_struct = mat_cnt[cnt_key]
    
    data = cnt_struct.x.T
    sfreq = float(cnt_struct.fs)
    ch_names = [str(ch).strip() for ch in cnt_struct.clab]
    
    if np.max(np.abs(data)) > 1.0:
        data = data * 1e-6
    
    if sfreq != SFREQ_EEG:
        print(f"Resampling from {sfreq} to {SFREQ_EEG} Hz")
        from scipy.signal import resample
        n_samples = int(data.shape[1] * SFREQ_EEG / sfreq)
        data = resample(data, n_samples, axis=1)
        sfreq = SFREQ_EEG
    
    info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types='eeg')
    raw = mne.io.RawArray(data, info, verbose=False)
    
    eog_channels = [ch for ch in ['VEOG', 'HEOG'] if ch in ch_names]
    if eog_channels:
        raw.set_channel_types({ch: 'eog' for ch in eog_channels})
    
    raw.set_montage('standard_1005', match_case=False, on_missing='ignore')
    
    mat_mrk = sio.loadmat(mrk_path, struct_as_record=False, squeeze_me=True)
    real_mrk_keys = [k for k in mat_mrk.keys() if not k.startswith('__')]
    mrk_key = real_mrk_keys[0]
    mrk_struct = mat_mrk[mrk_key]
    
    if hasattr(mrk_struct, 'time'):
        event_positions = np.round(mrk_struct.time * sfreq / 1000).astype(int)
    elif hasattr(mrk_struct, 'pos'):
        event_positions = np.array(mrk_struct.pos, dtype=int) - 1
    else:
        raise ValueError("could not find 'time' or 'pos' in marker struct")
    
    if hasattr(mrk_struct, 'event') and hasattr(mrk_struct.event, 'desc'):
        desc = mrk_struct.event.desc
        if desc.ndim > 1:
            desc = desc.squeeze()
        event_codes = np.asarray(desc, dtype=int).flatten()
    elif hasattr(mrk_struct, 'y'):
        y = mrk_struct.y
        if y.ndim > 1:
            if y.shape[0] == 1 or y.shape[1] == 1:
                event_codes = y.flatten()
            else:
                event_codes = np.argmax(y, axis=0) + 1
        else:
            event_codes = y.flatten()
    else:
        event_codes = event_positions.copy()
    
    events = np.zeros((len(event_positions), 3), dtype=int)
    events[:, 0] = event_positions
    events[:, 2] = event_codes
    
    print(f"EEG loaded: {raw.n_times} samples, {sfreq} Hz, {len(events)} markers")
    print(f"unique markers: {np.unique(event_codes)}")
    
    mat_mnt = sio.loadmat(mnt_path, struct_as_record=False, squeeze_me=True)
    real_mnt_keys = [k for k in mat_mnt.keys() if not k.startswith('__')]
    eeg_mnt_struct = mat_mnt[real_mnt_keys[0]]
    print(f"EEG mnt_nback.mat loaded: pos_3d shape {np.asarray(eeg_mnt_struct.pos_3d).shape}")
    
    return raw, events, sfreq, eeg_mnt_struct

# LOAD FNIRS DATA 
def load_fnirs_data(subject_id):
    fnirs_folder = os.path.join(BASE_DATASET_PATH, f"{subject_id}-NIRS")
    
    cnt_path = os.path.join(fnirs_folder, 'cnt_nback.mat')
    mnt_path = os.path.join(fnirs_folder, 'mnt_nback.mat')
    
    if not os.path.exists(cnt_path):
        raise FileNotFoundError(f"fnIRS file not found: {cnt_path}")
    
    print(f"Loading raw fNIRS from: {cnt_path}")
    
    mat_cnt = sio.loadmat(cnt_path, struct_as_record=False, squeeze_me=True)
    real_cnt_keys = [k for k in mat_cnt.keys() if not k.startswith('__')]
    cnt = mat_cnt[real_cnt_keys[0]]
    
    mat_mnt = sio.loadmat(mnt_path, struct_as_record=False, squeeze_me=True)
    real_mnt_keys = [k for k in mat_mnt.keys() if not k.startswith('__')]
    mnt = mat_mnt[real_mnt_keys[0]]
    
    sfreq_fnirs_raw = float(cnt.oxy.fs) if hasattr(cnt, 'oxy') and hasattr(cnt.oxy, 'fs') else 10.0
    print(f"raw fNIRS sampling rate: {sfreq_fnirs_raw} Hz")
    
    oxy_raw = np.asarray(cnt.oxy.x, dtype=np.float64)
    deoxy_raw = np.asarray(cnt.deoxy.x, dtype=np.float64)
    
    print(f"Raw fNIRS: oxy {oxy_raw.shape}, deoxy {deoxy_raw.shape}")
    
    print("Converting to HbO/HbR...")
    hbo_raw, hbr_raw = calculate_hbo_hbr(oxy_raw, deoxy_raw)
    
    print("Wavelet denoising...")
    hbo_wv = wavelet_denoising(hbo_raw)
    hbr_wv = wavelet_denoising(hbr_raw)
    
    print("MARA...")
    hbo_mara = apply_mara(hbo_wv, fs=sfreq_fnirs_raw)
    hbr_mara = apply_mara(hbr_wv, fs=sfreq_fnirs_raw)
    
    print("Low-pass filter (0.2 Hz)...")
    hbo_filtered = lowpass_filter(hbo_mara, fs=sfreq_fnirs_raw, cutoff=0.2)
    hbr_filtered = lowpass_filter(hbr_mara, fs=sfreq_fnirs_raw, cutoff=0.2)
    
    if sfreq_fnirs_raw != SFREQ_FNIRS:
        print(f"Resampling from {sfreq_fnirs_raw} Hz to {SFREQ_FNIRS} Hz...")
        from scipy.signal import resample
        n_samples = int(hbo_filtered.shape[0] * SFREQ_FNIRS / sfreq_fnirs_raw)
        hbo_resampled = resample(hbo_filtered, n_samples, axis=0)
        hbr_resampled = resample(hbr_filtered, n_samples, axis=0)
    else:
        hbo_resampled = hbo_filtered
        hbr_resampled = hbr_filtered
    
    print("Spatial mapping to 16x16...")
    hbo_mapped = spatial_mapping(hbo_resampled, mnt)
    hbr_mapped = spatial_mapping(hbr_resampled, mnt)
    
    fnirs_data = np.stack([hbo_mapped, hbr_mapped], axis=-1)
    
    print(f"fNIRS preprocessed: {fnirs_data.shape}")
    
    return fnirs_data

# FNIRS HELPER FUNCTIONS
def calculate_hbo_hbr(oxy, deoxy):
    lambda1, lambda2 = 760, 850
    L = 3.0
    
    def get_dpf(lmbda, age=25):
        alpha, beta, gamma = 223.3, 0.05624, 0.8493
        delta, eps, zeta = -5.723e-7, 0.001245, -0.9025
        return alpha + beta * (age ** gamma) + delta * (lmbda ** 3) + eps * (lmbda ** 2) + zeta * lmbda
    
    dpf1 = get_dpf(lambda1)
    dpf2 = get_dpf(lambda2)
    
    e1_hbo, e1_hbr = 0.586, 1.54852
    e2_hbo, e2_hbr = 1.058, 0.69132
    
    denom = L * (e1_hbr * e2_hbo - e2_hbr * e1_hbo)
    hbo = (e1_hbr * (deoxy / dpf2) - e2_hbr * (oxy / dpf1)) / denom
    hbr = (e1_hbo * (oxy / dpf1) - e2_hbo * (deoxy / dpf2)) / denom
    
    return hbo, hbr

def wavelet_denoising(data, alpha=0.1, wavelet='db5'):
    cleaned = np.zeros_like(data)
    n_samples, n_channels = data.shape
    
    for ch in range(n_channels):
        signal = data[:, ch]
        level = min(int(np.floor(np.log2(n_samples))), 6)
        coeffs = pywt.wavedec(signal, wavelet, level=level)
        new_coeffs = [coeffs[0]]
        for i in range(1, len(coeffs)):
            d_coeffs = coeffs[i].copy()
            sigma = np.median(np.abs(d_coeffs - np.median(d_coeffs))) / 0.6745
            if sigma > 0:
                threshold = sigma * stats.norm.ppf(1 - alpha / 2)
                d_coeffs[np.abs(d_coeffs) > threshold] = 0
            new_coeffs.append(d_coeffs)
        cleaned[:, ch] = pywt.waverec(new_coeffs, wavelet)[:n_samples]
    
    return cleaned

def apply_mara(data, fs=10.0, window_size=41, threshold_factor=1.8):
    n_samples, n_channels = data.shape
    corrected = np.zeros_like(data)
    edge_len = int(0.5 * fs)
    
    for ch in range(n_channels):
        signal = data[:, ch].copy()
        msd = pd.Series(signal).rolling(window=window_size, center=True, min_periods=1).std().to_numpy()
        threshold = np.median(msd) + (threshold_factor * np.std(msd))
        ma_mask = (msd > threshold).astype(int)
        
        diff_mask = np.diff(ma_mask, prepend=0, append=0)
        starts = np.where(diff_mask == 1)[0]
        ends = np.where(diff_mask == -1)[0]
        indices = sorted(list(set([0, n_samples] + list(starts) + list(ends))))
        segments = [signal[indices[i]:indices[i+1]].copy() for i in range(len(indices) - 1)]
        
        if not segments:
            corrected[:, ch] = signal
            continue
        
        final_parts = [segments[0]]
        for i in range(1, len(segments)):
            prev_seg = final_parts[-1]
            curr_seg = segments[i]
            val_prev = np.median(prev_seg[-edge_len:]) if len(prev_seg) >= edge_len else prev_seg[-1]
            val_curr = np.median(curr_seg[:edge_len]) if len(curr_seg) >= edge_len else curr_seg[0]
            shift = val_prev - val_curr
            
            if ma_mask[indices[i]] == 1:
                x = np.arange(len(curr_seg))
                s_val = max(np.var(curr_seg) * len(curr_seg), 1e-6)
                try:
                    spline = UnivariateSpline(x, curr_seg, s=s_val)
                    curr_seg = curr_seg - spline(x)
                    val_curr_new = np.median(curr_seg[:edge_len]) if len(curr_seg) >= edge_len else curr_seg[0]
                    curr_seg += (val_prev - val_curr_new)
                except:
                    curr_seg += shift
            else:
                if abs(shift) > 2 * np.std(msd):
                    curr_seg += shift
            final_parts.append(curr_seg)
        
        full_signal = np.concatenate(final_parts)[:n_samples]
        t_idx = np.arange(n_samples)
        original_trend = np.polyval(np.polyfit(t_idx, data[:, ch], 1), t_idx)
        corrected_trend = np.polyval(np.polyfit(t_idx, full_signal, 1), t_idx)
        full_signal += (original_trend - corrected_trend)
        corrected[:, ch] = full_signal
    
    return corrected

def lowpass_filter(data, fs=10.0, cutoff=0.2):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(3, normal_cutoff, btype='low')
    filtered = np.zeros_like(data)
    for ch in range(data.shape[1]):
        filtered[:, ch] = filtfilt(b, a, data[:, ch])
    return filtered

# SPATIAL MAPPING
def spatial_mapping(data, mnt):
    n_samples, n_channels = data.shape
    pos_3d = np.asarray(mnt.pos_3d, dtype=np.float64)
    x3d, y3d, z3d = pos_3d[0, :], pos_3d[1, :], pos_3d[2, :]
    r = np.sqrt(x3d**2 + y3d**2 + z3d**2)
    r_safe = np.where(r > 0, r, 1.0)
    theta = np.arccos(np.clip(z3d / r_safe, -1.0, 1.0))
    phi = np.arctan2(y3d, x3d)
    x_coords = theta * np.cos(phi)
    y_coords = theta * np.sin(phi)
    
    grid_x, grid_y = np.mgrid[min(x_coords):max(x_coords):16j, 
                               min(y_coords):max(y_coords):16j]
    points = np.column_stack((x_coords, y_coords))
    
    tri = Delaunay(points)
    values = data.T
    interp = CloughTocher2DInterpolator(tri, values, fill_value=0)
    result = interp((grid_x, grid_y))
    mapped = np.transpose(result, (2, 0, 1))
    
    return mapped

class _EEGMontageAdapter:
    def __init__(self, pos_3d):
        self.pos_3d = pos_3d

def get_eeg_mnt(raw, eeg_mnt_struct):
    full_clab = [str(c).strip() for c in np.asarray(eeg_mnt_struct.clab).flatten()]
    full_pos_3d = np.asarray(eeg_mnt_struct.pos_3d, dtype=np.float64)
    name_to_idx = {name: i for i, name in enumerate(full_clab)}
    name_to_idx_upper = {name.upper(): i for i, name in enumerate(full_clab)}

    pos_list = []
    missing = []
    for ch in raw.ch_names:
        idx = name_to_idx.get(ch, name_to_idx_upper.get(ch.upper(), None))
        if idx is None:
            missing.append(ch)
            pos_list.append([0.0, 0.0, 0.0])
        else:
            pos_list.append(full_pos_3d[:, idx])

    if missing:
        print(f"WARNINg: {len(missing)} EEG channel(s) not found in mnt_nback.mat's clab "
              f"(spatial_mapping will place them at the grid center): {missing}")

    pos_3d = np.array(pos_list, dtype=np.float64).T
    return _EEGMontageAdapter(pos_3d)

# PREPROCESS EEG
def preprocess_eeg(raw, events):
    print("Preprocessing EEG...")
    
    print("Bandpass filter: 0.5-50 Hz...")
    raw_filtered = raw.copy().filter(l_freq=0.5, h_freq=50.0, method='fir', verbose=False)
    
    raw_filtered.notch_filter(freqs=50.0, picks='eeg', verbose=False)
    
    print("Common Average Reference...")
    raw_filtered.set_eeg_reference('average', projection=False, verbose=False)
    
    print("ICA for artifact removal...")
    picks_for_ica = mne.pick_types(raw_filtered.info, eeg=True, eog=False, exclude='bads')
    ica = mne.preprocessing.ICA(n_components=0.99, method='picard',
                                fit_params=dict(ortho=False, extended=True),
                                random_state=42, verbose=False)
    ica.fit(raw_filtered, picks=picks_for_ica)
    
    try:
        ic_labels = mne_icalabel.label_components(raw_filtered, ica, method='iclabel')
        ARTIFACT_TARGETS = ['eye blink', 'eye movement', 'muscle artifact', 'line noise']
        artifact_idx = [idx for idx, label in enumerate(ic_labels['labels']) 
                        if label in ARTIFACT_TARGETS]
    except Exception as e:
        print(f"ICLabel failed: {e}, using fallback method...")
        artifact_idx = []
    
    try:
        if 'VEOG' in raw.ch_names:
            eog_indices, _ = ica.find_bads_eog(raw, ch_name='VEOG', threshold=2.0, verbose=False)
            artifact_idx.extend(eog_indices)
        if 'HEOG' in raw.ch_names:
            eog_indices, _ = ica.find_bads_eog(raw, ch_name='HEOG', threshold=2.0, verbose=False)
            artifact_idx.extend(eog_indices)
    except Exception as e:
        print(f"EOG detection failed: {e}")
    
    artifact_idx = sorted(set(artifact_idx))
    
    if len(artifact_idx) > 0:
        ica.exclude = artifact_idx
        ica.apply(raw_filtered, verbose=False)
        print(f"Removed {len(artifact_idx)} ICA components")
    else:
        print("No artifact components detected")
    
    channels_to_drop = [ch for ch in raw_filtered.ch_names if ch.upper() in ['VEOG', 'HEOG', 'TP9']]
    if channels_to_drop:
        raw_filtered.drop_channels(channels_to_drop)
        print(f"Dropped channels: {channels_to_drop}")
    
    return raw_filtered

# WINDOW EXTRACTION FOR TSFNet 
def extract_tsfnet_windows(eeg_raw, eeg_spatial_data, fnirs_data, events, sfreq_eeg):
    print("Extracting TSFNet windows...")
    print(f"EEG window: {EEG_WINDOW_LEN}s, step: {EEG_STEP}s")
    print(f"fNIRS segments: {FNIRS_NUM_SEGMENTS} segments of {FNIRS_WINDOW_LEN}s")
    
    label_map = EEG_TRIAL_LABEL_MAP
    valid_codes = list(label_map.keys())
    target_events = events[np.isin(events[:, 2], valid_codes)]
    
    print(f"Found {len(target_events)} trial-onset markers")
    
    if len(target_events) == 0:
        print(f"No trial-onset markers found. Available markers: {np.unique(events[:, 2])}")
        return np.array([]), np.array([]), np.array([])
    
    window_samples_eeg = int(EEG_WINDOW_LEN * sfreq_eeg)
    window_samples_fnirs = int(FNIRS_WINDOW_LEN * SFREQ_FNIRS)
    
    start_slide = EEG_EPOCH_TMIN
    end_slide = EEG_EPOCH_TMAX - EEG_WINDOW_LEN
    
    eeg_windows = []
    fnirs_windows = []
    labels = []
    
    for ev in target_events:
        event_sample_eeg = ev[0]
        raw_code = ev[2]
        
        label = label_map.get(raw_code, -1)
        if label == -1:
            continue
        
        event_time = event_sample_eeg / sfreq_eeg
        
        for offset_sec in np.arange(start_slide, end_slide + 0.001, EEG_STEP):
            
            w_start_eeg = int(event_sample_eeg + np.round(offset_sec * sfreq_eeg))
            w_end_eeg = w_start_eeg + window_samples_eeg
            
            if w_end_eeg > eeg_raw.n_times or w_start_eeg < 0:
                continue
            
            fnirs_segments = []
            valid_segments = True
            
            for seg_idx in range(FNIRS_NUM_SEGMENTS):
                seg_offset = offset_sec + seg_idx * FNIRS_STEP
                
                w_start_fnirs = int(np.round((event_time + seg_offset) * SFREQ_FNIRS))
                w_end_fnirs = w_start_fnirs + window_samples_fnirs
                
                if w_end_fnirs > fnirs_data.shape[0] or w_start_fnirs < 0:
                    valid_segments = False
                    break
                
                fnirs_slice = fnirs_data[w_start_fnirs:w_end_fnirs]
                fnirs_segments.append(fnirs_slice)
            
            if not valid_segments:
                continue
            
            eeg_slice_raw = eeg_raw.get_data(start=w_start_eeg, stop=w_end_eeg)
            if np.max(np.abs(eeg_slice_raw)) > 120e-6:
                continue
            
            eeg_slice_spatial = eeg_spatial_data[w_start_eeg:w_end_eeg]
            eeg_slice = eeg_slice_spatial.transpose(1, 2, 0)
            
            # Correct fNIRS shape for TSFNet
            fnirs_slice = np.stack(fnirs_segments, axis=0)  # (11, 30, 16, 16, 2)
            fnirs_slice = fnirs_slice.transpose(0, 2, 3, 1, 4)  # (11, 16, 16, 30, 2) 
            
            # Use float32 to save space
            eeg_windows.append(eeg_slice.astype(np.float32))
            fnirs_windows.append(fnirs_slice.astype(np.float32))
            labels.append(label)
    
    eeg_windows = np.array(eeg_windows, dtype=np.float32)
    fnirs_windows = np.array(fnirs_windows, dtype=np.float32)
    labels = np.array(labels, dtype=np.int32)
    
    print(f"Extracted {len(labels)} TSFNet windows")
    if len(labels) > 0:
        print(f"Label distribution: {np.bincount(labels)}")
        print(f"EEG windows shape: {eeg_windows.shape}")
        print(f"fNIRS windows shape: {fnirs_windows.shape}")
    
    return eeg_windows, fnirs_windows, labels

# LOGGING
LOG_FILE = os.path.join(OUTPUT_PATH, "processing_log.txt")

def log_message(msg):
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{timestamp}] {msg}"
    print(line)
    os.makedirs(OUTPUT_PATH, exist_ok=True)
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(line + "\n")

# MAIN PROCESSING FUNCTION
def process_subject(subject_id):
    t0 = time.time()
    log_message(f"START processing {subject_id}")
    print(f"\n{'='*60}")
    print(f"Processing {subject_id}")
    print(f"{'='*60}")
    
    eeg_folder = os.path.join(BASE_DATASET_PATH, f"{subject_id}-EEG")
    
    if not os.path.exists(eeg_folder):
        log_message(f"SKIPPED {subject_id}: EEG folder not found")
        print(f"EEG folder not found: {eeg_folder}")
        return
    
    try:
        print("\n[1] Loading EEG...")
        raw_eeg, events, sfreq_eeg, eeg_mnt_struct = load_eeg_data(eeg_folder)
        
        print("\n[2] Loading and preprocessing fNIRS...")
        fnirs_data = load_fnirs_data(subject_id)
        
        print("\n[3] Preprocessing EEG...")
        raw_eeg_clean = preprocess_eeg(raw_eeg, events)
        
        print("\n[4] Spatial mapping EEG to 16x16...")
        eeg_mnt = get_eeg_mnt(raw_eeg_clean, eeg_mnt_struct)
        eeg_full_data = raw_eeg_clean.get_data()
        eeg_spatial_full = spatial_mapping(eeg_full_data.T, eeg_mnt)
        print(f"EEG spatially mapped: {eeg_spatial_full.shape}")
        
        print("\n[5] Extracting TSFNet windows...")
        eeg_windows, fnirs_windows, labels = extract_tsfnet_windows(
            raw_eeg_clean, eeg_spatial_full, fnirs_data, events, sfreq_eeg
        )
        
        if len(labels) == 0:
            log_message(f"WARNING {subject_id}: no windows extracted")
            print(f"No windows extracted for {subject_id}")
            return
        
        print("\n[6] Saving raw preprocessed data (no normalization)...")
        sub_out_dir = os.path.join(OUTPUT_PATH, subject_id)
        os.makedirs(sub_out_dir, exist_ok=True)
        
        np.save(os.path.join(sub_out_dir, f"{subject_id}_eeg_windows.npy"), eeg_windows)
        np.save(os.path.join(sub_out_dir, f"{subject_id}_fnirs_windows.npy"), fnirs_windows)
        np.save(os.path.join(sub_out_dir, f"{subject_id}_labels.npy"), labels)
        
        metadata = {
            'subject_id': subject_id,
            'eeg_shape': list(eeg_windows.shape),
            'fnirs_shape': list(fnirs_windows.shape),
            'labels_distribution': np.bincount(labels).tolist(),
            'eeg_window_len': EEG_WINDOW_LEN,
            'eeg_step': EEG_STEP,
            'fnirs_num_segments': FNIRS_NUM_SEGMENTS,
            'fnirs_window_len': FNIRS_WINDOW_LEN,
            'sfreq_eeg': SFREQ_EEG,
            'sfreq_fnirs': SFREQ_FNIRS,
            'normalization': 'NONE'
        }
        with open(os.path.join(sub_out_dir, f"{subject_id}_metadata.json"), 'w') as f:
            json.dump(metadata, f, indent=2)
        
        print(f"OK {subject_id} completed!")
        print(f"EEG windows: {eeg_windows.shape}")
        print(f"fNIRS windows: {fnirs_windows.shape}")
        print(f"Labels: {np.bincount(labels)}")
        
        elapsed = (time.time() - t0) / 60
        log_message(f"DONE {subject_id} in {elapsed:.1f} min | "
                   f"EEG: {eeg_windows.shape} | fNIRS: {fnirs_windows.shape} | "
                   f"Labels: {np.bincount(labels)}")
        
        # Free memory after each subject
        del raw_eeg, events, raw_eeg_clean, eeg_spatial_full, eeg_windows, fnirs_windows, labels
        gc.collect()
        
    except Exception as e:
        elapsed = (time.time() - t0) / 60
        log_message(f"ERROR processing {subject_id} after {elapsed:.1f} min: {str(e)}")
        print(f"Error processing {subject_id}: {str(e)}")
        import traceback
        traceback.print_exc()

# RUN
if __name__ == "__main__":
    total_t0 = time.time()
    log_message("===== TSFNet PREPROCESSING PIPELINE STARTED =====")
    print("TSFNet PREPROCESSING PIPELINE (No Normalization)")
    print(f"EEG window: {EEG_WINDOW_LEN}s, step: {EEG_STEP}s")
    print(f"fNIRS segments: {FNIRS_NUM_SEGMENTS} segments of {FNIRS_WINDOW_LEN}s")
    print(f"fNIRS sampling rate: {SFREQ_FNIRS} Hz")
    print(f"Output path: {OUTPUT_PATH}")
    
    # Process all 26 subjects with memory cleanup
    for sub_num in range(13, 27):
        subject_id = f"VP{sub_num:03d}"
        process_subject(subject_id)
        gc.collect()
    
    print("ALL SUBJECTS PROCESSED")
    log_message("TSFNet PREPROCESSING PIPELINE COMPLETED")

[2026-07-07 22:07:39] ===== TSFNet PREPROCESSING PIPELINE STARTED =====
TSFNet PREPROCESSING PIPELINE (No Normalization)
EEG window: 3.0s, step: 1.0s
fNIRS segments: 11 segments of 3.0s
fNIRS sampling rate: 10.0 Hz
Output path: E:\kntu\Thesis\codes\Results\MULTIMODAL_PREPROCESSED
[2026-07-07 22:07:39] START processing VP013

Processing VP013

[1] Loading EEG...
loading EEG from: E:\kntu\Thesis\codes\DATA\VP013-EEG\cnt_nback.mat
EEG loaded: 388433 samples, 200.0 Hz, 567 markers
unique markers: [ 16  48  64  80  96 112 128 144]
EEG mnt_nback.mat loaded: pos_3d shape (3, 30)

[2] Loading and preprocessing fNIRS...
Loading raw fNIRS from: E:\kntu\Thesis\codes\DATA\VP013-NIRS\cnt_nback.mat
raw fNIRS sampling rate: 10.0 Hz
Raw fNIRS: oxy (20733, 36), deoxy (20733, 36)
Converting to HbO/HbR...
Wavelet denoising...
MARA...
Low-pass filter (0.2 Hz)...
Spatial mapping to 16x16...
fNIRS preprocessed: (20733, 16, 16, 2)

[3] Preprocessing EEG...
Preprocessing EEG...
Bandpass filter: 0.5-50 Hz...
C

In [2]:
# Model (needs to be checked and debugged)

import os
import gc
import random
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import warnings
warnings.filterwarnings('ignore')

# CONFIGURATION
PROCESSED_DATA_PATH = r"E:\kntu\Thesis\codes\Results\MULTIMODAL_PREPROCESSED"
OUTPUT_MODEL_PATH = r"E:\kntu\Thesis\codes\Results\TSFNET_MODEL"

os.makedirs(OUTPUT_MODEL_PATH, exist_ok=True)

# Clean temp files
import glob
pattern = os.path.join(PROCESSED_DATA_PATH, "t_mmap_*.npy")
for f in glob.glob(pattern):
    try:
        os.remove(f)
    except:
        pass

ALL_SUBJECTS = list(range(1, 27))

# BUG FIX: RANDOM_SEED was used below (torch.manual_seed(RANDOM_SEED), np.random.seed(RANDOM_SEED))
# but never defined anywhere in the file, causing an immediate NameError on the very first run.
# 42 is used here since it's the value this same project's other scripts have consistently used
# for the subject-split/model-init seed.
RANDOM_SEED = 42  # BUG FIX: was missing entirely

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

shuffled = ALL_SUBJECTS.copy()
random.shuffle(shuffled)

n_total = len(shuffled)
n_train = int(n_total * 0.70)
n_val = int(n_total * 0.15)

TRAIN_SUBS = shuffled[:n_train]
VAL_SUBS = shuffled[n_train:n_train + n_val]
TEST_SUBS = shuffled[n_train + n_val:]

print(f"\nSubject split:")
print(f"Train: {len(TRAIN_SUBS)}: {TRAIN_SUBS}")
print(f"Val:   {len(VAL_SUBS)}: {VAL_SUBS}")
print(f"Test:  {len(TEST_SUBS)}: {TEST_SUBS}")

# DATASET CLASS memory efficient version
class TSFNetDataset(Dataset):
    def __init__(self, subs_list, dataset_name="Dataset"):
        super().__init__()
        self.dataset_name = dataset_name
        self.total_windows = 0
        self.metadata = []
        self.cache = {}
        # self.cache_size = 1
        # PERFORMANCE BUG FIX: with cache_size=1 and shuffle=True across many subjects (18 in the
        # train split), nearly every __getitem__ call evicts the one cached subject and reloads a
        # completely different subject's FULL .npy files from disk via np.load() -- not lazy/memmap
        # access, but eager loading of the entire array into RAM every time. With each subject's
        # combined EEG+fNIRS files around ~330MB (11-segment fNIRS shape), and shuffle causing
        # most consecutive samples within a batch to belong to different subjects, this can mean
        # several GB of redundant disk I/O per single small batch -- almost certainly the actual
        # cause of the ~12s/batch, ~1hr/epoch slowdown observed. Raising cache_size to cover the
        # WHOLE split means every subject is loaded from disk at most ONCE per training run (after
        # which every subsequent access is a fast in-RAM cache hit), regardless of shuffle order.
        # SECOND BUG FIX (found via traceback: ZeroDivisionError in train_epoch): the line
        # `self.cache_size = len(self.metadata)` was placed HERE, immediately after
        # `self.metadata = []` above -- meaning self.metadata was STILL EMPTY at this point (the
        # loop that actually populates it runs further below), so cache_size silently became 0,
        # not 18. With cache_size=0, the very first _load_subject() call hit
        # `len(self.cache) >= self.cache_size` (0 >= 0, True) while self.cache was still empty,
        # so `next(iter(self.cache))` raised StopIteration. PyTorch's DataLoader iterator
        # machinery silently interprets a StopIteration raised inside __getitem__ as "iteration
        # finished", so the training for-loop ran ZERO times with no visible error until the
        # ZeroDivisionError on `correct / total` at the end of train_epoch. Fixed by moving this
        # assignment to AFTER the loop below, once self.metadata actually holds all subjects.
        
        print(f"\nLoading {dataset_name}...")
        
        for sub_id in subs_list:
            sub_str = f"{sub_id:03d}"
            subject = f"VP{sub_str}"
            sub_folder = os.path.join(PROCESSED_DATA_PATH, subject)
            
            if not os.path.exists(sub_folder):
                continue
            
            eeg_path = os.path.join(sub_folder, f"{subject}_eeg_windows.npy")
            fnirs_path = os.path.join(sub_folder, f"{subject}_fnirs_windows.npy")
            labels_path = os.path.join(sub_folder, f"{subject}_labels.npy")
            
            if os.path.exists(eeg_path) and os.path.exists(fnirs_path) and os.path.exists(labels_path):
                try:
                    labels = np.load(labels_path).astype(np.int64)
                    n_windows = labels.shape[0]
                    
                    self.metadata.append({
                        'subject': subject,
                        'eeg_path': eeg_path,
                        'fnirs_path': fnirs_path,
                        'labels_path': labels_path,
                        'n_windows': n_windows,
                        'start_idx': self.total_windows
                    })
                    
                    self.total_windows += n_windows
                    print(f"  {subject}: {n_windows} windows")
                    del labels
                    gc.collect()
                    
                except Exception as e:
                    print(f"  Error: {subject} - {e}")
                    continue
        
        if self.total_windows == 0:
            raise ValueError(f"No data found for {dataset_name}!")
        
        # BUG FIX (see comment above where this was previously, incorrectly, set too early):
        # self.metadata is now fully populated, so len(self.metadata) correctly reflects the
        # actual number of subjects successfully loaded in this split (e.g. 18 for train), not 0.
        self.cache_size = len(self.metadata)  # BUG FIX: moved here, now caches ALL subjects in this split simultaneously
        
        self.window_to_subject = []
        for idx, meta in enumerate(self.metadata):
            self.window_to_subject.extend([idx] * meta['n_windows'])
        
        print(f"{dataset_name} Ready! Windows: {self.total_windows}")
    
    def _load_subject(self, sub_idx):
        if sub_idx in self.cache:
            return self.cache[sub_idx]
        
        if len(self.cache) >= self.cache_size:
            old_key = next(iter(self.cache))
            del self.cache[old_key]
            gc.collect()
        
        meta = self.metadata[sub_idx]
        
        data = {
            'eeg': np.load(meta['eeg_path']).astype(np.float32),
            'fnirs': np.load(meta['fnirs_path']).astype(np.float32),
            'labels': np.load(meta['labels_path']).astype(np.int64)
        }
        
        self.cache[sub_idx] = data
        return data
    
    def __len__(self):
        return self.total_windows
    
    def __getitem__(self, idx):
        sub_idx = self.window_to_subject[idx]
        meta = self.metadata[sub_idx]
        local_idx = idx - meta['start_idx']
        
        data = self._load_subject(sub_idx)
        
        eeg = torch.from_numpy(data['eeg'][local_idx]).float()
        fnirs = torch.from_numpy(data['fnirs'][local_idx]).float()
        label = torch.tensor(data['labels'][local_idx], dtype=torch.long)
        
        return eeg, fnirs, label
    
    def cleanup(self):
        self.cache = {}
        gc.collect()


# POSITIONAL ENCODING 
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=11):
        super().__init__()
        self.pos_encoding = nn.Parameter(torch.randn(1, max_len, d_model) * 0.01)
        
    def forward(self, x):
        seq_len = x.size(1)
        return x + self.pos_encoding[:, :seq_len, :]


# CROSS ATTENTION 
class CrossAttention(nn.Module):

    def __init__(self, d_model, nhead=4, dropout=0.5):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, query, key, value):
        attn_out, _ = self.attn(query, key, value)
        return self.norm(query + self.dropout(attn_out))


# TSFNet MODEL 
class TSFNet(nn.Module):

    def __init__(self, num_classes=3, dropout=0.5):
        super(TSFNet, self).__init__()
        
        # EEG BRANCH 
        # Input: (batch, 1, 600, 16, 16)
        self.eeg_conv1 = nn.Conv3d(1, 16, kernel_size=(12, 4, 4), 
                                   stride=(6, 2, 2), padding=(3, 1, 1))
        self.eeg_bn1 = nn.BatchNorm3d(16)
        self.eeg_elu1 = nn.ELU(inplace=True)
        self.eeg_drop1 = nn.Dropout(dropout)
        
        self.eeg_conv2 = nn.Conv3d(16, 32, kernel_size=(6, 2, 2), 
                                   stride=(2, 2, 2), padding=(2, 0, 0))
        self.eeg_bn2 = nn.BatchNorm3d(32)
        self.eeg_elu2 = nn.ELU(inplace=True)
        self.eeg_drop2 = nn.Dropout(dropout)
        
        # fNIRS BRANCH 
        # Input: (batch, 2, 11, 16, 16)
        self.fnirs_conv1 = nn.Conv3d(2, 16, kernel_size=(4, 4, 4), 
                                     stride=(2, 2, 2), padding=(1, 1, 1))
        self.fnirs_bn1 = nn.BatchNorm3d(16)
        self.fnirs_elu1 = nn.ELU(inplace=True)
        self.fnirs_drop1 = nn.Dropout(dropout)
        
        self.fnirs_conv2 = nn.Conv3d(16, 32, kernel_size=(2, 2, 2), 
                                     stride=(2, 2, 2), padding=(0, 0, 0))
        self.fnirs_bn2 = nn.BatchNorm3d(32)
        self.fnirs_elu2 = nn.ELU(inplace=True)
        self.fnirs_drop2 = nn.Dropout(dropout)
        
        # FEATURE PROJECTIONS
        self.eeg_fc = nn.Linear(32 * 4 * 4, 64)
        
        self.fnirs_fc = nn.Linear(32 * 2, 64)
        
        # EFGF LAYER - EEG-fNIRS Guided Fusion 
        
        self.efgf_gamma = nn.Parameter(torch.tensor(0.5))
        self.efgf_alpha = nn.Parameter(torch.tensor(0.5))
        
        # Attention map generation 
        self.efgf_conv_eeg = nn.Conv2d(32, 1, kernel_size=4, stride=1, padding=0)
        self.efgf_conv_fnirs = nn.Conv3d(32, 1, kernel_size=(1, 4, 4), stride=1, padding=0)
        
        # CAFE LAYER - Cross-Attention 
        # BUG FIX: nirs_proj (used in forward() to project the fNIRS sequence to 64-d before the
        # CAFE cross-attention) was previously created INSIDE forward() as
        # nn.Linear(32, 64).to(device), a brand-new randomly-initialized layer on EVERY forward
        # call. Since it was a local variable, it was never registered in self.parameters(), so
        # the optimizer never updated it -- it injected fresh random noise into the CAFE block on
        # every single forward pass (train and eval alike) instead of learning anything. Moved
        # here to __init__ so it becomes a persistent, trainable layer like every other layer.
        self.fnirs_proj = nn.Linear(32, 64)  # BUG FIX: moved from forward(), now persistent & trainable
        self.pos_encoder = PositionalEncoding(64, max_len=11)
        self.cafe_attn1 = CrossAttention(64, nhead=4, dropout=dropout)
        self.cafe_attn2 = CrossAttention(64, nhead=4, dropout=dropout)
        self.cafe_gate = nn.Linear(128, 64)
        self.cafe_drop = nn.Dropout(dropout)
        
        # CLASSIFIER 
        self.fc_eeg = nn.Linear(64, num_classes)
        self.fc_fnirs = nn.Linear(64, num_classes)
        self.fc_fusion = nn.Linear(64, num_classes)
        
        self.w_eeg = nn.Parameter(torch.tensor(1.0))
        self.w_fnirs = nn.Parameter(torch.tensor(1.0))
        self.w_fusion = nn.Parameter(torch.tensor(1.0))
        
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, eeg, fnirs):
        batch_size = eeg.size(0)
        
        # EEG BRANCH
        eeg = eeg.permute(0, 3, 1, 2).unsqueeze(1)
        
        eeg_feat = self.eeg_conv1(eeg)
        eeg_feat = self.eeg_bn1(eeg_feat)
        eeg_feat = self.eeg_elu1(eeg_feat)
        eeg_feat = self.eeg_drop1(eeg_feat)
        
        eeg_feat = self.eeg_conv2(eeg_feat)
        eeg_feat = self.eeg_bn2(eeg_feat)
        eeg_feat = self.eeg_elu2(eeg_feat)
        eeg_feat = self.eeg_drop2(eeg_feat)
        
        # Global Average Pooling over temporal dimension
        eeg_feat_gap = eeg_feat.mean(dim=2)
        eeg_feat_flat = eeg_feat_gap.flatten(1)  
        eeg_feat_vec = self.eeg_fc(eeg_feat_flat)
        
        # fNIRS BRANCH
        fnirs = fnirs.permute(0, 5, 1, 2, 3, 4)
        fnirs = fnirs.mean(dim=-1)
        
        fnirs_feat = self.fnirs_conv1(fnirs)
        fnirs_feat = self.fnirs_bn1(fnirs_feat)
        fnirs_feat = self.fnirs_elu1(fnirs_feat)
        fnirs_feat = self.fnirs_drop1(fnirs_feat)
        
        fnirs_feat = self.fnirs_conv2(fnirs_feat)
        fnirs_feat = self.fnirs_bn2(fnirs_feat)
        fnirs_feat = self.fnirs_elu2(fnirs_feat)
        fnirs_feat = self.fnirs_drop2(fnirs_feat)
        
        # Global Average Pooling over spatial dimensions
        fnirs_feat_gap = fnirs_feat.mean(dim=(-1, -2))
        fnirs_feat_flat = fnirs_feat_gap.flatten(1)
        fnirs_feat_vec = self.fnirs_fc(fnirs_feat_flat) 
        
        # EFGF LAYER         
        # EEG attention
        eeg_attn = self.efgf_conv_eeg(eeg_feat_gap)
        eeg_attn = eeg_attn.squeeze(-1).squeeze(-1)
        
        # fNIRS attention
        fnirs_attn = self.efgf_conv_fnirs(fnirs_feat)
        # fnirs_attn = fnirs_attn.squeeze(-1).squeeze(-1).squeeze(-1)
        # BUG FIX: efgf_conv_fnirs's output has shape (batch, 1, D, 1, 1) where D is whatever is
        # left of the segment/temporal axis after the fnirs conv stack (verified D=2 here, NOT 1).
        # squeeze(-1) only removes dimensions that are already size 1; the third squeeze(-1) above
        # was therefore a silent no-op (D=2 is not squeezed), leaving fnirs_attn shaped
        # (batch, 1, 2) instead of the intended (batch, 1) to match eeg_attn. Combining these two
        # different-rank tensors below (phi = alpha*fnirs_attn + (1-alpha)*eeg_attn) does NOT raise
        # an error -- PyTorch's broadcasting rules silently produce a (batch, batch, 2) tensor (a
        # meaningless batch-by-batch outer-product-like result) instead of the intended per-sample
        # (batch, 1) attention scalar. Fixed by explicitly averaging over the leftover axis first.
        fnirs_attn = fnirs_attn.mean(dim=2)  # BUG FIX: (batch,1,D,1,1) -> (batch,1,1,1), collapses the leftover axis instead of leaving it unsquozen
        fnirs_attn = fnirs_attn.squeeze(-1).squeeze(-1)  # BUG FIX: (batch,1,1,1) -> (batch,1), now matches eeg_attn's shape exactly
        
        eeg_attn = torch.sigmoid(eeg_attn)
        fnirs_attn = torch.sigmoid(fnirs_attn)
        
        alpha = torch.clamp(self.efgf_alpha, 0, 1)
        phi = alpha * fnirs_attn + (1 - alpha) * eeg_attn
        
        gamma = torch.clamp(self.efgf_gamma, 0, 1)
        fused_feat = gamma * eeg_feat_vec + (1 - gamma) * fnirs_feat_vec + phi * fnirs_feat_vec
        
        # CAFE LAYER 
        # Prepare fNIRS sequence
        fnirs_seq = fnirs_feat_gap  
        fnirs_seq = fnirs_seq.permute(0, 2, 1)  
        
        # fnirs_proj = nn.Linear(32, 64).to(fnirs_seq.device)
        # BUG FIX: was re-created every forward call with random, untrained weights (see __init__ comment)
        fnirs_seq = self.fnirs_proj(fnirs_seq)  # BUG FIX: uses the persistent, trainable layer defined in __init__
        
        fnirs_seq = fnirs_seq.repeat(1, 6, 1)  
        fnirs_seq = fnirs_seq[:, :11, :]  
        fnirs_seq = self.pos_encoder(fnirs_seq)
        
        # Prepare fusion sequence
        fused_seq = fused_feat.unsqueeze(1) 
        fused_seq = fused_seq.repeat(1, 11, 1) 
        
        # Cross-attention 1: fNIRS attends to fusion features
        enhanced_fnirs = self.cafe_attn1(fnirs_seq, fused_seq, fused_seq)
        
        # Cross-attention 2: fusion attends to enhanced fNIRS
        fused_seq_avg = fused_seq.mean(dim=1, keepdim=True) 
        enhanced_fusion = self.cafe_attn2(fused_seq_avg, enhanced_fnirs, enhanced_fnirs)
        enhanced_fusion = enhanced_fusion.squeeze(1) 
        
        # Gated fusion
        gate_input = torch.cat([fused_feat, enhanced_fusion], dim=-1) 
        gate = torch.sigmoid(self.cafe_gate(gate_input))  
        final_feat = gate * enhanced_fusion + (1 - gate) * fused_feat 
        final_feat = self.cafe_drop(final_feat)
        
        # CLASSIFIER 
        eeg_out = self.fc_eeg(self.dropout(eeg_feat_vec))
        fnirs_out = self.fc_fnirs(self.dropout(fnirs_feat_vec))
        fusion_out = self.fc_fusion(self.dropout(final_feat))
        
        w = F.softmax(torch.stack([self.w_eeg, self.w_fnirs, self.w_fusion]), dim=0)
        final_out = w[0] * eeg_out + w[1] * fnirs_out + w[2] * fusion_out
        
        return final_out, eeg_out, fnirs_out, fusion_out, eeg_feat_gap, fnirs_feat, enhanced_fusion, enhanced_fnirs, phi


# LOSS FUNCTIONS 
class LabelSmoothingCrossEntropy(nn.Module):
    def __init__(self, smoothing=0.1):
        super().__init__()
        self.smoothing = smoothing
        
    def forward(self, pred, target):
        n_classes = pred.size(-1)
        with torch.no_grad():
            true_dist = torch.zeros_like(pred)
            true_dist.fill_(self.smoothing / (n_classes - 1))
            true_dist.scatter_(1, target.unsqueeze(1), 1 - self.smoothing)
        return torch.mean(torch.sum(-true_dist * F.log_softmax(pred, dim=-1), dim=-1))


class EFGFRegularizationLoss(nn.Module):

    def __init__(self):
        super().__init__()
        
    def pearson_correlation(self, u, v):
        # BUG FIX: u.view()/v.view() require the underlying tensor memory to already be
        # contiguous; v here is often the output of .expand_as() (a non-contiguous broadcasted
        # view), which raises "view size is not compatible with input tensor's size and stride"
        # the first time this loss is actually reached (previously unreachable due to bugs 1-3
        # above crashing/corrupting the forward pass before execution got this far).
        # .reshape() handles both contiguous and non-contiguous inputs safely (copying only when
        # necessary), fixing this without changing any numerical behavior.
        u_flat = u.reshape(u.size(0), -1)  # BUG FIX: .view() -> .reshape()
        v_flat = v.reshape(v.size(0), -1)  # BUG FIX: .view() -> .reshape()
        u_mean = u_flat - u_flat.mean(dim=1, keepdim=True)
        v_mean = v_flat - v_flat.mean(dim=1, keepdim=True)
        numerator = (u_mean * v_mean).sum(dim=1)
        denominator = torch.sqrt((u_mean ** 2).sum(dim=1) * (v_mean ** 2).sum(dim=1) + 1e-8)
        return numerator / denominator
    
    def forward(self, eeg_feat_map, attn_map):
        attn_map_expanded = attn_map.unsqueeze(-1).unsqueeze(-1)
        attn_map_expanded = attn_map_expanded.expand_as(eeg_feat_map)
        corr = self.pearson_correlation(eeg_feat_map, attn_map_expanded)
        return 1 - torch.abs(corr).mean()


class CAFERegularizationLoss(nn.Module):

    def __init__(self):
        super().__init__()
        
    def pearson_correlation(self, u, v):
        # BUG FIX: u.view()/v.view() require the underlying tensor memory to already be
        # contiguous; v here is often the output of .expand_as() (a non-contiguous broadcasted
        # view), which raises "view size is not compatible with input tensor's size and stride"
        # the first time this loss is actually reached (previously unreachable due to bugs 1-3
        # above crashing/corrupting the forward pass before execution got this far).
        # .reshape() handles both contiguous and non-contiguous inputs safely (copying only when
        # necessary), fixing this without changing any numerical behavior.
        u_flat = u.reshape(u.size(0), -1)  # BUG FIX: .view() -> .reshape()
        v_flat = v.reshape(v.size(0), -1)  # BUG FIX: .view() -> .reshape()
        u_mean = u_flat - u_flat.mean(dim=1, keepdim=True)
        v_mean = v_flat - v_flat.mean(dim=1, keepdim=True)
        numerator = (u_mean * v_mean).sum(dim=1)
        denominator = torch.sqrt((u_mean ** 2).sum(dim=1) * (v_mean ** 2).sum(dim=1) + 1e-8)
        return numerator / denominator
    
    def forward(self, enhanced_fnirs, enhanced_fusion):
        enhanced_fusion_expanded = enhanced_fusion.unsqueeze(1).expand_as(enhanced_fnirs)
        corr = self.pearson_correlation(enhanced_fnirs, enhanced_fusion_expanded)
        return 1 - torch.abs(corr).mean()


class TSFNetLoss(nn.Module):

    def __init__(self, lambda_efgf=0.2, lambda_cafe=0.2, smoothing=0.1):
        super().__init__()
        self.class_loss = LabelSmoothingCrossEntropy(smoothing=smoothing)
        self.efgf_loss = EFGFRegularizationLoss()
        self.cafe_loss = CAFERegularizationLoss()
        self.lambda_efgf = lambda_efgf
        self.lambda_cafe = lambda_cafe
        
    def forward(self, pred, target, eeg_feat_map, attn_map, enhanced_fnirs, enhanced_fusion):
        loss_class = self.class_loss(pred, target)
        loss_efgf = self.efgf_loss(eeg_feat_map, attn_map)
        loss_cafe = self.cafe_loss(enhanced_fnirs, enhanced_fusion)
        total_loss = loss_class + self.lambda_efgf * loss_efgf + self.lambda_cafe * loss_cafe
        return total_loss, loss_class, loss_efgf, loss_cafe


# METRICS
def accuracy_score(y_true, y_pred):
    return np.mean(np.array(y_true) == np.array(y_pred))

def cohen_kappa_score(y1, y2):
    y1 = np.array(y1)
    y2 = np.array(y2)
    classes = np.unique(np.concatenate([y1, y2]))
    n = len(classes)
    cm = np.zeros((n, n), dtype=np.int64)
    class_to_idx = {c: i for i, c in enumerate(classes)}
    for i in range(len(y1)):
        cm[class_to_idx[y1[i]], class_to_idx[y2[i]]] += 1
    po = np.trace(cm) / np.sum(cm)
    pe = 0
    for i in range(n):
        pe += (np.sum(cm[i, :]) * np.sum(cm[:, i])) / (np.sum(cm) ** 2)
    return (po - pe) / (1 - pe + 1e-8)

def confusion_matrix(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    classes = np.unique(np.concatenate([y_true, y_pred]))
    n = len(classes)
    cm = np.zeros((n, n), dtype=np.int64)
    class_to_idx = {c: i for i, c in enumerate(classes)}
    for i in range(len(y_true)):
        cm[class_to_idx[y_true[i]], class_to_idx[y_pred[i]]] += 1
    return cm


# TRAINING FUNCTIONS
def train_epoch(model, loader, optimizer, loss_fn, device):
    model.train()
    total_loss = 0.0
    total_class_loss = 0.0
    total_efgf_loss = 0.0
    total_cafe_loss = 0.0
    correct = 0
    total = 0
    
    for eeg_b, fnirs_b, labels_b in loader:
        eeg_b = eeg_b.to(device, non_blocking=True)
        fnirs_b = fnirs_b.to(device, non_blocking=True)
        labels_b = labels_b.to(device, non_blocking=True)
        
        optimizer.zero_grad()
        
        pred, eeg_out, fnirs_out, fusion_out, eeg_feat_gap, fnirs_feat, enhanced_fusion, enhanced_fnirs, phi = model(eeg_b, fnirs_b)
        
        loss, loss_class, loss_efgf, loss_cafe = loss_fn(
            pred, labels_b, eeg_feat_gap, phi, enhanced_fnirs, enhanced_fusion
        )
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        total_loss += loss.item()
        total_class_loss += loss_class.item()
        total_efgf_loss += loss_efgf.item()
        total_cafe_loss += loss_cafe.item()
        correct += (pred.argmax(1) == labels_b).sum().item()
        total += labels_b.size(0)
        
        del eeg_b, fnirs_b, labels_b, pred, loss
        
    return (total_loss / len(loader), 
            total_class_loss / len(loader),
            total_efgf_loss / len(loader),
            total_cafe_loss / len(loader),
            correct / total)


def evaluate(model, loader, device):
    model.eval()
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for eeg_b, fnirs_b, labels_b in loader:
            eeg_b = eeg_b.to(device, non_blocking=True)
            fnirs_b = fnirs_b.to(device, non_blocking=True)
            labels_b = labels_b.to(device, non_blocking=True)
            
            pred, _, _, _, _, _, _, _, _ = model(eeg_b, fnirs_b)
            
            correct += (pred.argmax(1) == labels_b).sum().item()
            total += labels_b.size(0)
            
            all_preds.extend(pred.argmax(1).cpu().numpy())
            all_labels.extend(labels_b.cpu().numpy())
            
            del eeg_b, fnirs_b, labels_b, pred
    
    return correct / total, all_preds, all_labels


# MAIN EXECUTION
if __name__ == "__main__":
        
    # LOAD DATASETS
    print("\n[1] Loading datasets...")
    train_dataset = TSFNetDataset(TRAIN_SUBS, "Train")
    val_dataset = TSFNetDataset(VAL_SUBS, "Validation")
    test_dataset = TSFNetDataset(TEST_SUBS, "Test")
    
    BATCH_SIZE = 16
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, 
                              num_workers=0, pin_memory=False)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, 
                            num_workers=0, pin_memory=False)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, 
                             num_workers=0, pin_memory=False)
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # INITIALIZE MODEL
    print(f"\n[2] Initializing model...")
    print(f" Device: {device}")
    print(f"Batch size: {BATCH_SIZE}")
    
    model = TSFNet(num_classes=3, dropout=0.5).to(device)
    
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")
    
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', 
                                                      factor=0.5, patience=10)
    
    loss_fn = TSFNetLoss(lambda_efgf=0.2, lambda_cafe=0.2, smoothing=0.1)
    
    # TRAINING LOOP
    EPOCHS = 300
    EARLY_STOP_PATIENCE = 50
    
    print(f"\n[4] Starting training...")
    print(f"Epochs: {EPOCHS}")
    print(f"Early stopping patience: {EARLY_STOP_PATIENCE}")
    
    best_val_acc = 0.0
    epochs_no_improve = 0
    best_epoch = 0
    
    for epoch in range(1, EPOCHS + 1):
        train_loss, class_loss, efgf_loss, cafe_loss, train_acc = train_epoch(
            model, train_loader, optimizer, loss_fn, device
        )
        val_acc, _, _ = evaluate(model, val_loader, device)
        
        scheduler.step(val_acc)
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch
            torch.save(model.state_dict(), os.path.join(OUTPUT_MODEL_PATH, 'best_tsfnet.pth'))
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
        
        if epoch % 10 == 0 or epoch == 1:
            print(f"Epoch {epoch:3d}/{EPOCHS} | "
                  f"Loss: {train_loss:.4f} | "
                  f"Class: {class_loss:.4f} | "
                  f"EFGF: {efgf_loss:.4f} | "
                  f"CAFE: {cafe_loss:.4f} | "
                  f"Train: {train_acc*100:.2f}% | "
                  f"Val: {val_acc*100:.2f}% | "
                  f"Best: {best_val_acc*100:.2f}%")
        
        if epochs_no_improve >= EARLY_STOP_PATIENCE:
            print(f"\nEarly stopping at epoch {epoch}")
            print(f"Best validation accuracy: {best_val_acc*100:.2f}% at epoch {best_epoch}")
            break
        
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()
    
    # TEST EVALUATION
    print("[5] Final Test Evaluation")
    
    model.load_state_dict(torch.load(os.path.join(OUTPUT_MODEL_PATH, 'best_tsfnet.pth')))
    model.eval()
    
    test_acc, test_preds, test_labels = evaluate(model, test_loader, device)
    
    class_names = ['0-back', '2-back', '3-back']
    
    print(f"\nTest Accuracy: {test_acc*100:.2f}%")
    print(f"Kappa Score: {cohen_kappa_score(test_labels, test_preds):.4f}")
    
    cm = confusion_matrix(test_labels, test_preds)
    print("\nConfusion Matrix:")
    print("             0-back  2-back  3-back")
    for i, row in enumerate(cm):
        print(f"  {class_names[i]:>8}  {row[0]:>6}  {row[1]:>6}  {row[2]:>6}")
    
    print("[6] Ablation Study")

    
    model.eval()
    all_preds_eeg = []
    all_preds_fnirs = []
    all_preds_fusion = []
    all_labels = []
    
    with torch.no_grad():
        for eeg_b, fnirs_b, labels_b in test_loader:
            eeg_b = eeg_b.to(device)
            fnirs_b = fnirs_b.to(device)
            
            _, eeg_out, fnirs_out, fusion_out, _, _, _, _, _ = model(eeg_b, fnirs_b)
            
            all_preds_eeg.extend(eeg_out.argmax(1).cpu().numpy())
            all_preds_fnirs.extend(fnirs_out.argmax(1).cpu().numpy())
            all_preds_fusion.extend(fusion_out.argmax(1).cpu().numpy())
            all_labels.extend(labels_b.cpu().numpy())
    
    print(f"\nEEG Branch Only:      {accuracy_score(all_labels, all_preds_eeg)*100:.2f}%")
    print(f"fNIRS Branch Only:    {accuracy_score(all_labels, all_preds_fnirs)*100:.2f}%")
    print(f"EFGF+CAFE Fusion:     {accuracy_score(all_labels, all_preds_fusion)*100:.2f}%")
    print(f"Full TSFNet:          {test_acc*100:.2f}%")
    
    with torch.no_grad():
        w = F.softmax(torch.stack([model.w_eeg, model.w_fnirs, model.w_fusion]), dim=0)
        print(f"\n  Learned weights (Eq. 4):")
        print(f"w_eeg = {w[0].item():.3f}")
        print(f"w_fnirs = {w[1].item():.3f}")
        print(f"w_fusion = {w[2].item():.3f}")
    
    # SAVE RESULTS
    results = {
        'test_accuracy': test_acc,
        'kappa': cohen_kappa_score(test_labels, test_preds),
        'confusion_matrix': cm.tolist(),
        'ablation': {
            'eeg_only': accuracy_score(all_labels, all_preds_eeg),
            'fnirs_only': accuracy_score(all_labels, all_preds_fnirs),
            'fusion_only': accuracy_score(all_labels, all_preds_fusion),
            'full_tsfnet': test_acc
        },
        'learned_weights': {
            'w_eeg': w[0].item(),
            'w_fnirs': w[1].item(),
            'w_fusion': w[2].item()
        },
        'best_epoch': best_epoch,
        'best_val_acc': best_val_acc
    }
    
    import json
    with open(os.path.join(OUTPUT_MODEL_PATH, 'results.json'), 'w') as f:
        json.dump(results, f, indent=2)
    
    print("TRAINING COMPLETE!")
    print(f"moodel saved to: {OUTPUT_MODEL_PATH}")
    print(f"Results saved to: {os.path.join(OUTPUT_MODEL_PATH, 'results.json')}")
    
    # CLEANUP
    print("\n[7] Cleaning up...")
    train_dataset.cleanup()
    val_dataset.cleanup()
    test_dataset.cleanup()
    
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    
    print("Cleanup completed.")


Subject split:
Train: 18: [3, 20, 21, 24, 18, 6, 23, 7, 11, 2, 16, 25, 17, 13, 22, 26, 14, 9]
Val:   3: [19, 4, 12]
Test:  5: [1, 15, 10, 5, 8]

[1] Loading datasets...

Loading Train...
  VP003: 270 windows
  VP020: 268 windows
  VP021: 270 windows
  VP024: 270 windows
  VP018: 270 windows
  VP006: 269 windows
  VP023: 270 windows
  VP007: 259 windows
  VP011: 270 windows
  VP002: 270 windows
  VP016: 263 windows
  VP025: 267 windows
  VP017: 270 windows
  VP013: 270 windows
  VP022: 270 windows
  VP026: 266 windows
  VP014: 270 windows
  VP009: 267 windows
Train Ready! Windows: 4829

Loading Validation...
  VP019: 257 windows
  VP004: 120 windows
  VP012: 270 windows
Validation Ready! Windows: 647

Loading Test...
  VP001: 246 windows
  VP015: 263 windows
  VP010: 265 windows
  VP005: 211 windows
  VP008: 270 windows
Test Ready! Windows: 1255

[2] Initializing model...
 Device: cuda
Batch size: 16
Total parameters: 105,008
Trainable parameters: 105,008

[4] Starting training...
Epoc